<a href="https://colab.research.google.com/github/isaiahdm792-alt/stock--screener/blob/main/01_full_scan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import requests
from io import StringIO # Import StringIO

def get_sp500_tickers():
    """Scrape the current S&P 500 constituent list from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

    # Wrap the response text in StringIO as suggested by the FutureWarning
    tables = pd.read_html(StringIO(response.text))
    sp500_table = tables[0]  # first table on the page is the constituent list
    tickers = sp500_table["Symbol"].tolist()
    # yfinance expects dots as hyphens for some tickers (e.g. BRK.B -> BRK-B)
    tickers = [t.replace(".", "-") for t in tickers]
    return tickers

sp500_tickers = get_sp500_tickers()
print(f"Pulled {len(sp500_tickers)} tickers.")
print(sp500_tickers[:10], "...")

Pulled 503 tickers.
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A'] ...


In [11]:
# STEP 1: Fundamentals Scanner — starter script
# Paste this into a Google Colab cell and run it.
# First run: install yfinance (only needed once per Colab session)

# !pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import time

# --- Start small: 15 well-known tickers to test the pipeline first ---
# Once this works cleanly, swap in the full S&P 500 list (step below).
test_tickers = sp500_tickers

def get_fundamentals(ticker):
    """Pull key fundamental ratios for one ticker. Returns a dict or None on failure."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        pe = info.get("trailingPE")
        ps = info.get("priceToSalesTrailing12Months")
        pb = info.get("priceToBook")
        peg = info.get("pegRatio")
        fcf = info.get("freeCashflow")
        earnings_growth = info.get("earningsGrowth")
        sector = info.get("sector")

        return {
            "ticker": ticker,
            "sector": sector,
            "pe_ratio": pe,
            "ps_ratio": ps,
            "pb_ratio": pb,
            "peg_ratio": peg,
            "free_cash_flow": fcf,
            "earnings_growth": earnings_growth,
        }
    except Exception as e:
        print(f"  [!] Failed on {ticker}: {e}")
        return None

# --- Run the scan ---
results = []
for t in test_tickers:
    print(f"Pulling {t}...")
    data = get_fundamentals(t)
    if data:
        results.append(data)
    time.sleep(0.5)  # be polite to the API, avoid rate-limit issues

df = pd.DataFrame(results)

# --- Basic cleanup: drop rows missing the core ratios ---
df_clean = df.dropna(subset=["pe_ratio", "ps_ratio"])

print("\n--- Results ---")
print(df_clean.sort_values("pe_ratio").to_string(index=False))

# --- Save to CSV so you can commit it to GitHub / inspect later ---
df_clean.to_csv("fundamentals_snapshot.csv", index=False)
print("\nSaved to fundamentals_snapshot.csv")

Pulling MMM...
Pulling AOS...
Pulling ABT...
Pulling ABBV...
Pulling ACN...
Pulling ADBE...
Pulling AMD...
Pulling AES...
Pulling AFL...
Pulling A...
Pulling APD...
Pulling ABNB...
Pulling AKAM...
Pulling ALB...
Pulling ARE...
Pulling ALGN...
Pulling ALLE...
Pulling LNT...
Pulling ALL...
Pulling GOOGL...
Pulling GOOG...
Pulling MO...
Pulling AMZN...
Pulling AMCR...
Pulling AEE...
Pulling AEP...
Pulling AXP...
Pulling AIG...
Pulling AMT...
Pulling AWK...
Pulling AMP...
Pulling AME...
Pulling AMGN...
Pulling APH...
Pulling ADI...
Pulling AON...
Pulling APA...
Pulling APO...
Pulling AAPL...
Pulling AMAT...
Pulling APP...
Pulling APTV...
Pulling ACGL...
Pulling ADM...
Pulling ARES...
Pulling ANET...
Pulling AJG...
Pulling AIZ...
Pulling T...
Pulling ATO...
Pulling ADSK...
Pulling ADP...
Pulling AZO...
Pulling AVB...
Pulling AVY...
Pulling AXON...
Pulling BKR...
Pulling BALL...
Pulling BAC...
Pulling BAX...
Pulling BDX...
Pulling BRK-B...
Pulling BBY...
Pulling TECH...
Pulling BIIB...
Pulli

In [ ]:
# STEP 2: Basic sector-relative scoring
import pandas as pd

df = df_clean.copy()  # use the results from Step 1

# Calculate the average P/E per sector
sector_avg_pe = df.groupby("sector")["pe_ratio"].transform("mean")

# Flag stocks trading below their sector average P/E
df["pe_vs_sector"] = df["pe_ratio"] - sector_avg_pe
df["undervalued_flag"] = df["pe_vs_sector"] < 0

# Simple composite score: lower PEG and lower relative P/E = higher score
df["score"] = (
    (1 / df["peg_ratio"].clip(lower=0.1)) * 40   # reward low PEG
    - df["pe_vs_sector"].clip(lower=0) * 0.5      # penalize high relative P/E
)

df_ranked = df.sort_values("score", ascending=False)

print(df_ranked[["ticker", "sector", "pe_ratio", "peg_ratio", "undervalued_flag", "score"]].head(20).to_string(index=False))

df_ranked.to_csv("scored_snapshot.csv", index=False)
print("\nSaved to scored_snapshot.csv")

In [12]:
# STEP 2: Technical Indicators — no extra libraries needed, avoids pandas-ta version conflicts
# Run this after your fundamentals scan (Part 1) is done and df_ranked exists.
# No pip installs needed for this version - just yfinance and pandas, which you already have.

import yfinance as yf
import pandas as pd
import time

def calculate_rsi(prices, period=14):
    """Manual RSI calculation - no external library needed."""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_macd(prices, fast=12, slow=26, signal=9):
    """Manual MACD calculation - no external library needed."""
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line

def get_technicals(ticker):
    """Pull price history and calculate SMA, RSI, MACD, and volume anomaly for one ticker."""
    try:
        hist = yf.Ticker(ticker).history(period="1y")
        if hist.empty or len(hist) < 200:
            return None  # not enough history to calculate 200-day SMA

        hist["sma50"] = hist["Close"].rolling(window=50).mean()
        hist["sma200"] = hist["Close"].rolling(window=200).mean()
        hist["rsi"] = calculate_rsi(hist["Close"])
        hist["macd"], hist["macd_signal"] = calculate_macd(hist["Close"])

        avg_volume_30d = hist["Volume"].tail(30).mean()
        latest = hist.iloc[-1]

        return {
            "ticker": ticker,
            "price": latest["Close"],
            "sma50": latest["sma50"],
            "sma200": latest["sma200"],
            "above_sma50": latest["Close"] > latest["sma50"],
            "above_sma200": latest["Close"] > latest["sma200"],
            "rsi": latest["rsi"],
            "macd_bullish": latest["macd"] > latest["macd_signal"],
            "volume_today": latest["Volume"],
            "avg_volume_30d": avg_volume_30d,
            "volume_spike": latest["Volume"] > (avg_volume_30d * 1.5),
        }
    except Exception as e:
        print(f"  [!] Failed on {ticker}: {e}")
        return None

# --- Test on a small slice first, same pattern as Step 1 ---
test_slice = sp500_tickers

tech_results = []
for t in test_slice:
    print(f"Pulling technicals for {t}...")
    data = get_technicals(t)
    if data:
        tech_results.append(data)
    time.sleep(0.5)

df_tech = pd.DataFrame(tech_results)
print("\n--- Technical Results ---")
print(df_tech.to_string(index=False))

df_tech.to_csv("technicals_snapshot.csv", index=False)
print("\nSaved to technicals_snapshot.csv")

Pulling technicals for MMM...
Pulling technicals for AOS...
Pulling technicals for ABT...
Pulling technicals for ABBV...
Pulling technicals for ACN...
Pulling technicals for ADBE...
Pulling technicals for AMD...
Pulling technicals for AES...
Pulling technicals for AFL...
Pulling technicals for A...
Pulling technicals for APD...
Pulling technicals for ABNB...
Pulling technicals for AKAM...
Pulling technicals for ALB...
Pulling technicals for ARE...
Pulling technicals for ALGN...
Pulling technicals for ALLE...
Pulling technicals for LNT...
Pulling technicals for ALL...
Pulling technicals for GOOGL...
Pulling technicals for GOOG...
Pulling technicals for MO...
Pulling technicals for AMZN...
Pulling technicals for AMCR...
Pulling technicals for AEE...
Pulling technicals for AEP...
Pulling technicals for AXP...
Pulling technicals for AIG...
Pulling technicals for AMT...
Pulling technicals for AWK...
Pulling technicals for AMP...
Pulling technicals for AME...
Pulling technicals for AMGN...
P

In [ ]:
# Normalize each sub-score to 0-100 before combining
def normalize(series):
    return (series - series.min()) / (series.max() - series.min()) * 100

df_combined["fundamentals_score"] = normalize(1 / df_combined["peg_ratio"].clip(lower=0.1))
df_combined["technicals_score"] = (
    df_combined["above_sma50"].astype(int) * 25
    + df_combined["above_sma200"].astype(int) * 25
    + df_combined["macd_bullish"].astype(int) * 25
    + df_combined["volume_spike"].astype(int) * 25
)  # already naturally 0-100

# Now combine with deliberate weights (adjust these as you like)
df_combined["score"] = (
    df_combined["fundamentals_score"] * 0.6
    + df_combined["technicals_score"] * 0.4
)

df_combined = df_combined.sort_values("score", ascending=False)
print(df_combined[["ticker", "fundamentals_score", "technicals_score", "score"]].head(20).to_string(index=False))

df_combined.to_csv("scored_snapshot.csv", index=False)

In [13]:
from google.colab import userdata
NEWSAPI_KEY = userdata.get('NEWSAPI_KEY')

In [15]:
!pip install transformers torch --quiet

In [16]:
# STEP 5: Sentiment layer
# Only run on your top candidates (not the full 500) due to NewsAPI's 100 requests/day free limit.
# Run Step 5.1 (pip install transformers torch) before this cell.

import requests
import pandas as pd
from transformers import pipeline

# --- Load FinBERT (downloads the model once, then caches it for this session) ---
print("Loading FinBERT model... this takes a minute the first time.")
sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")
print("Model loaded.")

def get_headlines(ticker, api_key, page_size=10):
    """Pull recent headlines for a ticker via NewsAPI."""
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": ticker,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": page_size,
        "apiKey": api_key,
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        if data.get("status") != "ok":
            print(f"  [!] NewsAPI error for {ticker}: {data.get('message')}")
            return []
        return [article["title"] for article in data.get("articles", []) if article.get("title")]
    except Exception as e:
        print(f"  [!] Failed to fetch news for {ticker}: {e}")
        return []

def score_sentiment(headlines):
    """Run headlines through FinBERT and return an average sentiment score from -1 to +1."""
    if not headlines:
        return None
    results = sentiment_pipeline(headlines)
    # FinBERT returns label: positive/negative/neutral, with a confidence score
    total = 0
    for r in results:
        if r["label"] == "positive":
            total += r["score"]
        elif r["label"] == "negative":
            total -= r["score"]
        # neutral contributes 0
    return total / len(results)

# --- Load your existing scored data, take the top N candidates ---
df_scored = pd.read_csv("scored_snapshot.csv")
top_candidates = df_scored.sort_values("score", ascending=False).head(50)["ticker"].tolist()

print(f"Running sentiment on top {len(top_candidates)} candidates...")

sentiment_results = []
for t in top_candidates:
    print(f"Fetching news for {t}...")
    headlines = get_headlines(t, NEWSAPI_KEY, page_size=10)
    sent_score = score_sentiment(headlines)
    sentiment_results.append({
        "ticker": t,
        "headline_count": len(headlines),
        "sentiment_score": sent_score,
    })

df_sentiment = pd.DataFrame(sentiment_results)
print("\n--- Sentiment Results ---")
print(df_sentiment.to_string(index=False))

df_sentiment.to_csv("sentiment_snapshot.csv", index=False)
print("\nSaved to sentiment_snapshot.csv")

Loading FinBERT model... this takes a minute the first time.


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded.
Running sentiment on top 50 candidates...
Fetching news for HIG...
Fetching news for CSGP...
Fetching news for MU...
Fetching news for GPN...
Fetching news for SW...
Fetching news for TPR...
Fetching news for DAL...
Fetching news for COF...
Fetching news for GM...
Fetching news for FIS...
Fetching news for MSFT...
Fetching news for MET...
Fetching news for CVS...
Fetching news for ZBRA...
Fetching news for AMCR...
Fetching news for ZBH...
Fetching news for APA...
Fetching news for CVX...
Fetching news for EME...
Fetching news for IR...
Fetching news for BKNG...
Fetching news for BEN...
Fetching news for IQV...
Fetching news for XYZ...
Fetching news for VZ...
Fetching news for CI...
Fetching news for IFF...
Fetching news for RJF...
Fetching news for EXPE...
Fetching news for CPAY...
Fetching news for SYF...
Fetching news for COP...
Fetching news for MDLZ...
Fetching news for KDP...
Fetching news for PYPL...
Fetching news for CBRE...
Fetching news for TSN...
Fetching news f

In [17]:
df_final = df_scored.merge(df_sentiment, on="ticker", how="left")
df_final["sentiment_norm"] = normalize(df_final["sentiment_score"].fillna(0))

df_final["score"] = (
    df_final["fundamentals_score"] * 0.45
    + df_final["technicals_score"] * 0.35
    + df_final["sentiment_norm"] * 0.20
)

df_final = df_final.sort_values("score", ascending=False)
print(df_final[["ticker", "fundamentals_score", "technicals_score", "sentiment_norm", "score"]].head(20).to_string(index=False))

df_final.to_csv("scored_snapshot.csv", index=False)

ticker  fundamentals_score  technicals_score  sentiment_norm     score
   HIG           91.660251                75       87.933570 85.083827
   GPN           45.791629                75       92.411876 65.338608
    SW           42.263273                75       74.313983 60.131270
    MU           84.603540                25       54.947301 57.811053
   COF           49.961503                50       85.559066 57.094490
  CSGP          100.000000                25       13.111907 56.372381
   TPR           36.617904                75       58.168277 54.361712
   FIS           45.791629                50       79.790116 54.064256
  CBRE           10.715624                75      100.000000 51.072031
    GM           31.375776                75       51.145650 50.598229
   DAL           52.344289                50       45.400443 50.135019
   FDS            8.946342                75       96.973685 49.670591
  AMCR           17.678602                75       69.155463 48.036464
  MSFT

In [19]:
# STEP 6a: Insider trading data via SEC EDGAR
# No API key needed - SEC EDGAR is fully public.
# Run this after your Step 5 (sentiment) work, on the same top-candidates list.

import requests
import pandas as pd
import time

# SEC requires a descriptive User-Agent header identifying who's making requests
# Replace with your own name/email - this is SEC's policy, not optional
HEADERS = {"User-Agent": "ANT - personal stock research project - davismooreisaiah@gmail.com"}

def get_cik(ticker):
    """Look up a company's SEC CIK number (their unique filer ID) from their ticker."""
    url = "https://www.sec.gov/cgi-bin/browse-edgar"
    params = {"action": "getcompany", "company": ticker, "type": "4", "dateb": "", "owner": "include", "count": "1", "output": "atom"}
    try:
        # SEC also provides a direct ticker-to-CIK mapping file, which is more reliable:
        lookup_url = "https://www.sec.gov/files/company_tickers.json"
        response = requests.get(lookup_url, headers=HEADERS, timeout=10)
        data = response.json()
        for entry in data.values():
            if entry["ticker"] == ticker:
                return str(entry["cik_str"]).zfill(10)  # CIK needs to be 10 digits, zero-padded
        return None
    except Exception as e:
        print(f"  [!] CIK lookup failed for {ticker}: {e}")
        return None

def get_insider_buys(ticker, cik, months_back=3):
    """Pull recent Form 4 filings for a company and count insider buy transactions
    actually filed within the last `months_back` months."""
    if not cik:
        return {"ticker": ticker, "recent_form4_filings": None}

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        data = response.json()
        recent = data.get("filings", {}).get("recent", {})
        forms = recent.get("form", [])
        filing_dates = recent.get("filingDate", [])

        cutoff = pd.Timestamp.now() - pd.DateOffset(months=months_back)

        # Only count Form 4s whose filing date actually falls within the cutoff window
        form4_count = sum(
            1 for f, d in zip(forms, filing_dates)
            if f == "4" and pd.Timestamp(d) >= cutoff
        )

        return {"ticker": ticker, "recent_form4_filings": form4_count}
    except Exception as e:
        print(f"  [!] Filing pull failed for {ticker}: {e}")
        return {"ticker": ticker, "recent_form4_filings": None}

# --- Run on your top candidates, same list as the sentiment step ---
df_scored = pd.read_csv("scored_snapshot.csv")
top_candidates = df_scored.sort_values("score", ascending=False).head(50)["ticker"].tolist()

insider_results = []
for t in top_candidates:
    print(f"Looking up {t}...")
    cik = get_cik(t)
    data = get_insider_buys(t, cik)
    insider_results.append(data)
    time.sleep(0.3)  # SEC asks for no more than ~10 requests/second, this is well within that

df_insider = pd.DataFrame(insider_results)
print("\n--- Insider Filing Results ---")
print(df_insider.to_string(index=False))

df_insider.to_csv("insider_snapshot.csv", index=False)
print("\nSaved to insider_snapshot.csv")

Looking up HIG...
Looking up GPN...
Looking up SW...
Looking up MU...
Looking up COF...
Looking up CSGP...
Looking up TPR...
Looking up FIS...
Looking up CBRE...
Looking up GM...
Looking up DAL...
Looking up FDS...
Looking up AMCR...
Looking up MSFT...
Looking up BLK...
Looking up ABBV...
Looking up EXPE...
Looking up ZBRA...
Looking up IQV...
Looking up CCL...
Looking up CPAY...
Looking up CI...
Looking up ZBH...
Looking up IFF...
Looking up APA...
Looking up BKNG...
Looking up SOLV...
Looking up IR...
Looking up CVS...
Looking up MDLZ...
Looking up CVX...
Looking up KDP...
Looking up XYZ...
Looking up LMT...
Looking up A...
Looking up MET...
Looking up REGN...
Looking up OXY...
Looking up SBUX...
Looking up WTW...
Looking up LH...
Looking up L...
Looking up XOM...
Looking up PRU...
Looking up CDW...
Looking up PCAR...
Looking up ABNB...
Looking up WAB...
Looking up BALL...
Looking up SWK...

--- Insider Filing Results ---
ticker  recent_form4_filings
   HIG                    12
   G

In [20]:
from google.colab import userdata
FRED_API_KEY = userdata.get('FRED_API_KEY')
print(f"Key loaded, starts with: {FRED_API_KEY[:4]}...")

Key loaded, starts with: ab63...


In [21]:
# STEP 6b: Macro data via FRED (Federal Reserve Economic Data)
# Run after confirming FRED_API_KEY is loaded (see setup above).

import requests
import pandas as pd

def get_fred_series(series_id, api_key, limit=1):
    """Pull the most recent value(s) for a FRED economic data series."""
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "sort_order": "desc",
        "limit": limit,
    }
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    return data.get("observations", [])

# --- Pull the key macro indicators ---
# FRED series IDs: FEDFUNDS = federal funds rate, CPIAUCSL = CPI (inflation), UNRATE = unemployment
fed_funds = get_fred_series("FEDFUNDS", FRED_API_KEY)
cpi = get_fred_series("CPIAUCSL", FRED_API_KEY, limit=13)  # 13 months to calculate YoY change
unemployment = get_fred_series("UNRATE", FRED_API_KEY)

current_fed_funds = float(fed_funds[0]["value"])
current_unemployment = float(unemployment[0]["value"])

# Calculate year-over-year inflation from CPI data
cpi_now = float(cpi[0]["value"])
cpi_year_ago = float(cpi[-1]["value"])
inflation_yoy = ((cpi_now - cpi_year_ago) / cpi_year_ago) * 100

print(f"Current Fed Funds Rate: {current_fed_funds}%")
print(f"Current Unemployment Rate: {current_unemployment}%")
print(f"Year-over-year Inflation (CPI): {inflation_yoy:.2f}%")

# --- Simple macro regime classification ---
# This is a simplified heuristic, not economic forecasting - a starting point you can refine
if current_fed_funds > 4.5:
    rate_regime = "high_rates"
elif current_fed_funds > 2:
    rate_regime = "moderate_rates"
else:
    rate_regime = "low_rates"

print(f"\nCurrent rate regime: {rate_regime}")

# Save this as a simple macro snapshot - it applies to ALL stocks equally today,
# unlike the per-stock data you've pulled so far
macro_snapshot = pd.DataFrame([{
    "date": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "fed_funds_rate": current_fed_funds,
    "unemployment_rate": current_unemployment,
    "inflation_yoy": inflation_yoy,
    "rate_regime": rate_regime,
}])

macro_snapshot.to_csv("macro_snapshot.csv", index=False)
print("\nSaved to macro_snapshot.csv")

Current Fed Funds Rate: 3.63%
Current Unemployment Rate: 4.2%
Year-over-year Inflation (CPI): 3.46%

Current rate regime: moderate_rates

Saved to macro_snapshot.csv


In [22]:
import pandas as pd

df_final = pd.read_csv("scored_snapshot.csv")
macro = pd.read_csv("macro_snapshot.csv").iloc[0]

# In a moderate/high rate environment, expensive growth stocks (high PEG) get a small penalty,
# and lower-PEG "value" stocks get a small bonus - reflecting that capital is costlier for growth right now
if macro["rate_regime"] in ["high_rates", "moderate_rates"]:
    # Stocks with low PEG get a small bonus, high PEG a small penalty
    df_final["macro_adjustment"] = df_final["fundamentals_score"].apply(
        lambda x: 3 if x > 60 else (-3 if x < 30 else 0)
    )
else:  # low_rates - growth gets a bit more benefit of the doubt
    df_final["macro_adjustment"] = df_final["technicals_score"].apply(
        lambda x: 3 if x > 60 else 0
    )

df_final["score"] = df_final["score"] + df_final["macro_adjustment"]
df_final = df_final.sort_values("score", ascending=False)

print(df_final[["ticker", "score", "macro_adjustment"]].head(20).to_string(index=False))
df_final.to_csv("scored_snapshot.csv", index=False)

ticker     score  macro_adjustment
   HIG 88.083827                 3
   GPN 65.338608                 0
    MU 60.811053                 3
    SW 60.131270                 0
  CSGP 59.372381                 3
   COF 57.094490                 0
   TPR 54.361712                 0
   FIS 54.064256                 0
    GM 50.598229                 0
   DAL 50.135019                 0
  CBRE 48.072031                -3
   FDS 46.670591                -3
  AMCR 45.036464                -3
  MSFT 44.311781                -3
   BLK 44.244739                -3
  ABBV 44.083591                -3
  EXPE 43.822682                -3
  ZBRA 43.695100                -3
   CVS 43.402934                 0
   IQV 43.198839                -3
